# Deep Learning Framework for Environmental Sound Recognition Using PyTorch
### Reproducible laboratory notebook · UrbanSound8K · Official PANNs CNN14 (AudioSet pretrained)

**Scope:** single-label classification of the ten official UrbanSound8K classes; partial fine-tuning of CNN14's later layers.  
**Evaluation:** predefined UrbanSound8K folds; 8 train / 1 validation / 1 reserved test.  
**Environment verified on student's Windows PC:** Python 3.13.15, PyTorch 2.7.0+cu128, RTX 3050 Laptop (4 GiB VRAM).  
**Evidence as of 23 October 2026:** Stage 2/3 full audit passed; Stage 4 one-step gradient/update checks passed for `head`, `fc1`, and `block6`. **Full training, held-out test metrics, CO4 and CO5 quantitative results are NOT completed.**

This **new** notebook is the current project; `environmental_sound_recognition.ipynb` from the earlier **ESC-50/SimpleCNN** experiment is historical and must not be cited as current performance.

---

### Navigation
1. Notebook controls, filesystem and durable experiment logs  
2. Stage 1 — project requirements and experimental protocol  
3. Stage 2 — Python/CUDA, dataset audit, official CNN14 provenance  
4. Stage 3 — UrbanSound8K statistics, preprocessing, source overlap, DataLoaders  
5. Stage 4 — architecture, freezing, actual gradient checks and optional training  
6. Training history and CO3/CO4/CO5 evidence board  
7. README synchronization, safe cache cleanup, reproducibility and next steps

> **How to operate from VS Code:** Open `notebooks/01_UrbanSound8K_PANNs_CNN14_Project.ipynb` and select the interpreter/kernel where `torch==2.7.0+cu128` works. Execute cells with **Shift+Enter**, or **Restart & Run All** after checking the switches below. **Save the notebook (Ctrl+S)** to retain its displayed tables, outputs and figures after shutdown. The notebook also writes dated logs and JSON to `results/notebook_evidence/`, independently of Jupyter autosave.
>
> The notebook uses project Python modules via `subprocess` for long checks/training, but you launch them by running notebook cells—not by typing commands into a terminal. This avoids copying the same neural-network implementation twice.
>
> **Fresh Git checkout:** Git does not contain your large UrbanSound8K audio or pretrained CNN14 checkpoint; restore them to `data/` and `checkpoints/` locally. The official PANNs source may be cloned through an opt-in cell below. None of these files is downloaded on every notebook restart.

## 0 · Notebook settings (read before Run All)

The default run is **safe and exploratory**: no installs, recloning, checkpoint deserialization of the legacy pickle, full audio scans, long gradient checks, or training. Switch on a selected task only when you intend to execute it. Results of previous Windows runs remain visible from stored logs even with all switches disabled.

In [ ]:
# ---------- USER CONTROLS: deliberately conservative defaults ----------
RUN_PACKAGE_CHECK = False          # run pip check and import audit*1
CLONE_PANNS_IF_MISSING = False     # explicit opt-in: small official Git repository
RUN_ORIGINAL_DATASET_AUDIT = False # 8,732 metadata/file path check
RUN_FULL_AUDIO_AUDIT = False       # decode all 8,732 clips + figures
RUN_UNIT_TESTS = False            # run the current unittest suite*1
RUN_CNN14_FORWARD = False         # load 1.4 GB tensor-only model + forward*1
RUN_REAL_STAGE4_GRADIENT_CHECKS = False # 3 real CUDA backward/update checks*1
RUN_DIAGNOSTIC_PILOTS = False     # short 4-batch train/val, NOT valid performance*2*3
RUN_FULL_FOLD_TRAINING = False    # genuine training; validation-selected; potentially long*3
TRAIN_CONFIGURATIONS = ('head', 'fc1', 'block6')
TEST_FOLD = 1
EPOCHS = 6
PATIENCE = 2
BATCH_SIZE = 1
ACCUM_STEPS = 4
AMP_MODE = 'auto'                  # BF16 when supported; otherwise FP32
UPDATE_README_FROM_NOTEBOOK = True
CLEAR_PYTHON_CACHES = False        # manual clean ONLY, never dataset/checkpoints/results

assert TEST_FOLD in range(1, 11)
assert all(c in {'head','fc1','block6'} for c in TRAIN_CONFIGURATIONS)
print('Configuration loaded. Expensive actions are opt-in.')

## 0.1 · Imports and project discovery

The location is determined from the notebook's working directory so it functions whether VS Code starts in the root folder or inside `notebooks/`. Only source-code identifiers are imported here; no CNN14 weights are loaded.

In [ ]:
from pathlib import Path
from datetime import datetime
from collections import deque
import os, sys, subprocess, shutil, hashlib, json, platform, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
ROOT = next((p for p in candidates if (p / 'src' / 'dataset.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the project folder in VS Code and extract Stage 3/4 sources into src/.')
sys.path.insert(0, str(ROOT))
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
EVIDENCE = ROOT / 'results' / 'notebook_evidence' / RUN_ID
EVIDENCE.mkdir(parents=True, exist_ok=True)
notebook_steps = []
print('Project root:', ROOT)
print('Interpreter:', sys.executable)
print('Evidence directory:', EVIDENCE)
print('Notebook outputs persist when the .ipynb file is SAVED.')

## 0.2 · Persistent runner: keep **complete** logs on disk

Heavy model checks and training are run by the current notebook kernel's Python executable. All stdout/stderr goes to a dated `.log` file, while the notebook shows a concise completion summary or relevant progress lines. Errors stop that cell and preserve the log; the manifest is written even for failed runs.

*For reliable long runs, keep VS Code and the notebook kernel open until the task completes. Intermediate training history and checkpoints are also written by `src.finetune_train`.*

In [ ]:
def run_logged(name, args, *, live=False, check=True):
    """Run repository Python code from a notebook, persist full output and exit status."""
    args = [str(part) for part in args]
    log_file = EVIDENCE / (''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in name) + '.log')
    print(f'▶ {name}:', subprocess.list2cmdline(args), flush=True)
    tail = deque(maxlen=24)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    # Capture stdout and stderr in a file. Avoid overwhelming notebook outputs
    # with 8,732 progress-bar refreshes from tqdm.
    with log_file.open('w', encoding='utf-8', buffering=1) as fp:
        process = subprocess.Popen(args, cwd=ROOT, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        for raw in process.stdout:
            fp.write(raw)
            for fragment in raw.replace('\r', '\n').splitlines():
                fragment = fragment.strip()
                if not fragment:
                    continue
                tail.append(fragment)
                if live and (fragment.startswith(('Epoch ', 'CONFIG=', 'RESULT:', 'Stage 4 check status:',
                                                  'Training run completed', 'FULL AUDIT',
                                                  'STAGE 4 ONE-STEP', 'ERROR', 'Traceback'))):
                    print(fragment, flush=True)
        rc = process.wait()
    record = {'name': name, 'command': args, 'exit_code': rc,
              'log': str(log_file.relative_to(ROOT)), 'recorded_at': datetime.now().isoformat()}
    notebook_steps.append(record)
    (EVIDENCE / 'steps.json').write_text(json.dumps(notebook_steps, indent=2), encoding='utf-8')
    print(f'Exit={rc}; complete log: {log_file.relative_to(ROOT)}')
    for line in list(tail)[-12:]:
        print('   ' + line[:280])
    if rc != 0 and check:
        raise RuntimeError(f'{name} failed. Inspect saved log: {log_file}')
    return record

print('Durable logging helper ready.')

## Stage 1 · Requirements, methodology and historical progress

**Project title must remain unchanged.** Official assignment: **CO3 Application (10 marks), CO4 Same-dataset comparative analysis (5 marks), CO5 experimental contribution (5 marks)**. The current innovation *candidate* is controlled analysis of which pretrained CNN14 layers to fine-tune; it is **not** a claim of an invented architecture or a proven performance improvement.

### Development timeline (reported Windows evidence, not notebook-executed output)

| Milestone | Local evidence | Verified result |
|:--|:--|:--|
| Teacher/project alignment | `project_submission_instructions` | UrbanSound8K + PANNs CNN14 is the current implementation; ESC-50 is history |
| Stage 2 dataset audit | `check_urbansound8k.py` | 8,732 WAV paths, 10 classes, 10 folds, 0 missing |
| Stage 2 checkpoint | official `Cnn14_mAP=0.431.pth`, MD5 `595633ac2d1cac7ef04ebf70e2fee4e4` | Hash matched; strictly loaded original 527-output model |
| Stage 2 official code | Git revision `d2f4b8c18eab44737fcc0de1248ae21eb43f6aa4` | Strict `state_dict` compatibility; safe tensor-only export |
| Stage 3 full audit | `results/verification_runs/20261023_011110_002188/` | 8,732/8,732 audio decoded, 0 invalid; real-audio CNN14 forward passed |
| Stage 3 split audit | `results/cross_fold_source_summary.csv` | Official clip partitions disjoint; **five source-ID overlaps warned** |
| Stage 4 unit tests and gradients | `results/stage4/checks/20261023_071516_424568/` | **19/19 tests**, `head`, `fc1`, `block6` real CUDA updates passed |
| Stage 4 learning/evaluation | — | **Not yet established**: no confirmed pilot/full epoch/held-out metrics |

Earlier engineering issues were resolved: incorrect upstream file-path assumption, legacy pickle incompatibility with restricted PyTorch loading, CUDA FP16 nonfinite logits, and a zero-gradient *single-clip* false alarm. That last issue remains an observation: the first dog-bark candidate yielded zero `fc1` input/gradient; another training clip yielded nonzero gradients and parameter updates.

**Do not interpret one-step gradients as model accuracy or convergence.**

## Stage 2 · Environment reproducibility and official sources

In [ ]:
import torch
import librosa
import soundfile as sf
import sklearn
from importlib import metadata as package_metadata
versions = {}
for pkg in ['torch','torchlibrosa','numpy','scipy','librosa','soundfile','pandas',
            'scikit-learn','matplotlib','seaborn','tqdm','ipykernel']:
    try: versions[pkg] = package_metadata.version(pkg)
    except package_metadata.PackageNotFoundError: versions[pkg] = 'MISSING'
device_info = {
    'python': sys.version.split()[0], 'executable': sys.executable,
    'platform': platform.platform(), 'cuda_available': torch.cuda.is_available(),
    'torch_cuda_build': torch.version.cuda,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_total_gib': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
                     if torch.cuda.is_available() else None,
    'versions': versions,
}
(EVIDENCE/'environment.json').write_text(json.dumps(device_info, indent=2), encoding='utf-8')
display(pd.DataFrame([{'package':k,'version':v} for k,v in versions.items()]))
print('CUDA:', device_info['cuda_available'], '|', device_info['gpu_name'],
      '| VRAM GiB:', device_info['gpu_total_gib'])

In [ ]:
if RUN_PACKAGE_CHECK:
    run_logged('pip_requirements_consistency', [sys.executable, '-m','pip','check'])
    run_logged('dependency_import_and_CUDA', [sys.executable, 'check_dependencies.py'])
else:
    print('Optional pip/CUDA recheck skipped. Results from prior verified Windows run are in the README.')

### Official PANNs source and pretrained checkpoint (provenance)

The verified reference is <https://github.com/qiuqiangkong/audioset_tagging_cnn> at commit `d2f4b8c18eab44737fcc0de1248ae21eb43f6aa4`. Use the original **Cnn14** AudioSet 527-class architecture. The **trusted legacy checkpoint** was checked and exported once to `checkpoints/Cnn14_AudioSet_state_dict_safe.pth`, which loads using `weights_only=True`. **Never automatically load arbitrary third-party pickle files with `weights_only=False` inside this notebook.**

Pretrained frontend settings: sample rate **32,000 Hz**, window **1024**, hop **320**, **64 Mel bins**, fmin **50 Hz**, fmax **14,000 Hz**; PANNs CNN14 computes its log-Mel representation internally.

In [ ]:
PANNS = ROOT / 'external' / 'audioset_tagging_cnn'
EXPECTED_COMMIT = 'd2f4b8c18eab44737fcc0de1248ae21eb43f6aa4'
if not (PANNS/'pytorch'/'models.py').is_file() and CLONE_PANNS_IF_MISSING:
    (ROOT/'external').mkdir(exist_ok=True)
    run_logged('clone_official_PANNs', ['git','clone','https://github.com/qiuqiangkong/audioset_tagging_cnn.git',str(PANNS)])
    run_logged('checkout_verified_PANNs_revision', ['git','-C',str(PANNS),'checkout','--detach',EXPECTED_COMMIT])
if (PANNS/'pytorch'/'models.py').is_file():
    found = subprocess.check_output(['git','-C',str(PANNS),'rev-parse','HEAD'],text=True).strip()
    print('Official PANNs revision:',found)
    print('Matches verified revision:',found == EXPECTED_COMMIT)
    if found != EXPECTED_COMMIT:
        print('WARNING: source revision mismatch; Stage 4 load_model will reject it. Do not silently change source.')
else:
    print('PANNs source absent. Set CLONE_PANNS_IF_MISSING=True and rerun this cell when connected.')

ORIGINAL_CKPT = ROOT/'checkpoints'/'Cnn14_mAP=0.431.pth'
SAFE_CKPT = ROOT/'checkpoints'/'Cnn14_AudioSet_state_dict_safe.pth'
if ORIGINAL_CKPT.is_file():
    with ORIGINAL_CKPT.open('rb') as chk_stream:
        md5 = hashlib.file_digest(chk_stream, 'md5').hexdigest()
    print('Original checkpoint MD5:', md5, '| official hash matches:',md5=='595633ac2d1cac7ef04ebf70e2fee4e4')
else:
    print('Original checkpoint missing (do not commit/download it automatically).')
print('Tensor-only safe checkpoint exists:',SAFE_CKPT.is_file())

## Stage 3 · Metadata and official ten-fold data protocol

UrbanSound8K's class IDs are **not alphabetically reindexed**. Each outer test fold uses the following official fold for validation; the remaining eight folds are training. The held-out test data must not be used to choose checkpoints, tuning values, architecture or freezing depth.

The five `fsID` values spanning folds are **potential source-related dependence**, not a reason to silently change the published folds. The original recording time intervals include positive cross-fold overlaps for some source IDs; see the detailed CSV audit.

In [ ]:
from src.dataset import (DATASET_ROOT, SAMPLE_RATE, TARGET_SAMPLES, CLASS_NAMES,
                         load_metadata, split_metadata, read_waveform, make_loaders)
META_FILE = DATASET_ROOT/'metadata'/'UrbanSound8K.csv'
if not META_FILE.is_file():
    print('DATA NOT FOUND:',META_FILE)
    print('Restore the official UrbanSound8K dataset under data/UrbanSound8K/.')
    df = None
else:
    df = load_metadata()
    train_df, val_df, test_df = split_metadata(df, test_fold=TEST_FOLD)
    print('Metadata validation PASS; clips:',len(df),'classes:',df.classID.nunique(),
          'folds:',df.fold.nunique())
    print('Test fold:', TEST_FOLD, '| validation fold:', TEST_FOLD % 10 + 1)
    print('Train / validation / reserved test:',len(train_df),len(val_df),len(test_df))
    display(pd.DataFrame({'classID':range(10),'official class name':CLASS_NAMES}))

In [ ]:
if df is not None:
    class_stats = (df.groupby(['classID','class']).size().rename('clips')
                   .reset_index().sort_values('classID'))
    class_stats['percent'] = (100*class_stats['clips']/len(df)).round(2)
    display(class_stats)
    fig,ax=plt.subplots(figsize=(11,4))
    ax.bar(class_stats['class'], class_stats['clips'])
    ax.set(title='UrbanSound8K: official class distribution',ylabel='Number of clips',xlabel='Class')
    ax.tick_params(axis='x',rotation=40)
    fig.tight_layout(); fig.savefig(EVIDENCE/'class_distribution.png',dpi=140)
    display(fig); plt.close(fig)
else:
    print('Dataset unavailable; histogram skipped.')

In [ ]:
if df is not None:
    fold_counts = df.groupby('fold').size().rename('clips').reset_index()
    fold_class = (df.pivot_table(index='fold',columns='classID',values='slice_file_name',
                                 aggfunc='count',fill_value=0).reindex(columns=range(10)))
    display(fold_counts)
    display(fold_class)
    fig,ax=plt.subplots(figsize=(9,4))
    m=ax.imshow(fold_class.to_numpy(),aspect='auto')
    ax.set(title='Clips per original UrbanSound8K fold and class',xlabel='Class ID',ylabel='Fold')
    ax.set_xticks(range(10),range(10))
    ax.set_yticks(range(10),range(1,11))
    fig.colorbar(m,ax=ax,label='Clips')
    fig.tight_layout();fig.savefig(EVIDENCE/'fold_class_matrix.png',dpi=140)
    display(fig);plt.close(fig)
    splits=[]
    for fold in range(1,11):
        tr,v,te=split_metadata(df,fold)
        splits.append({'test_fold':fold,'val_fold':fold%10+1,'train_clips':len(tr),
                       'val_clips':len(v),'test_clips':len(te),'total':len(df)})
    split_table=pd.DataFrame(splits)
    display(split_table)
    split_table.to_csv(EVIDENCE/'ten_fold_split_table.csv',index=False)
else:
    print('Metadata unavailable; fold analysis skipped.')

In [ ]:
# The source isolation warning is an existing DATASET property, not training leakage we introduce.
if df is not None:
    collisions = (df.groupby('fsID').agg(fold_count=('fold','nunique'),
                                         clips=('fold','size')).query('fold_count > 1'))
    if not collisions.empty:
        display(collisions)
        print('SOURCE-ISOLATION WARNING: shared fsID count:',len(collisions))
        print('Original fold assignments are unchanged.')
        detail = ROOT/'results'/'cross_fold_source_summary.csv'
        if detail.is_file():
            display(pd.read_csv(detail))
    else:
        print('No source-ID overlap found in this metadata.')
else:
    print('Metadata unavailable; source-overlap audit skipped.')
if RUN_ORIGINAL_DATASET_AUDIT:
    run_logged('dataset_path_audit', [sys.executable,'check_urbansound8k.py'])
    run_logged('official_fold_audit', [sys.executable,'src/check_folds.py'])

### Waveform preprocessing: inspect, visualize and explain

`read_waveform` uses `soundfile` decoding → arithmetic-mean mono conversion → resample to **32,000 Hz** (`soxr_hq`) as necessary → deterministic **center crop** of long clips / **right-zero padding** of short clips → `float32` vector with **128,000 samples**. Audio augmentation is **not** applied by the data pipeline; the official PANNs model applies training-mode SpecAugment and dropout internally. **Do not feed the illustrative Mel spectrogram below to CNN14:** it accepts raw waveforms.

In [ ]:
if df is not None:
    row=df.iloc[0]
    wav_file=DATASET_ROOT/'audio'/f"fold{int(row['fold'])}"/str(row['slice_file_name'])
    raw_info = sf.info(wav_file)
    waveform=read_waveform(wav_file)
    print('Example:',wav_file.name,'| class:',row['class'],f"(ID {int(row['classID'])})")
    print('Original sample rate:',raw_info.samplerate,'original duration (s):',round(raw_info.duration,4))
    print('Processed waveform shape:',waveform.shape,'dtype:',waveform.dtype,
          'min/max:',float(waveform.min()),float(waveform.max()))
    assert waveform.shape==(128000,) and waveform.dtype==np.float32
    fig,ax=plt.subplots(figsize=(10,2.5))
    ax.plot(np.arange(len(waveform))/SAMPLE_RATE,waveform,lw=.5)
    ax.set(title='Mono 32 kHz waveform (model input)',xlabel='Time (s)',ylabel='Amplitude')
    fig.tight_layout();fig.savefig(EVIDENCE/'waveform_example.png',dpi=140)
    display(fig);plt.close(fig)
else:
    print('Waveform demo skipped: data absent.')

In [ ]:
# Visualization for understanding the PANNs frontend, NOT a network input.
if df is not None:
    import librosa.display
    mel=librosa.feature.melspectrogram(y=waveform,sr=SAMPLE_RATE,n_fft=1024,
             hop_length=320,n_mels=64,fmin=50,fmax=14000,power=2.0)
    mel_db=librosa.power_to_db(mel,ref=np.max)
    fig,ax=plt.subplots(figsize=(10,3))
    im=librosa.display.specshow(mel_db,sr=SAMPLE_RATE,hop_length=320,x_axis='time',
                                 y_axis='mel',cmap='magma',ax=ax)
    ax.set(title='Educational log-Mel visualization (CNN14 computes its own)')
    fig.colorbar(im,ax=ax,format='%+d dB')
    fig.tight_layout();fig.savefig(EVIDENCE/'logmel_explanation.png',dpi=140)
    display(fig);plt.close(fig)
else:
    print('Mel explanation skipped: data absent.')

### Full-data integrity evidence (from Stage 3)

The prior **complete** Windows scan decoded all 8,732 WAV files with zero invalid recordings. Observed sample rates ranged from **8 kHz to 192 kHz**, and durations from **0.05 s to approximately 4.037 s**; eight exceeded 4.001 seconds. The notebook reads that actual JSON file when present and never fabricates a new audit.

In [ ]:
if RUN_FULL_AUDIO_AUDIT:
    run_logged('full_audio_decode_audit', [sys.executable,'-m','src.audit_audio'],live=True)
report_path=ROOT/'results'/'dataset_integrity_report.json'
if report_path.is_file():
    dataset_report=json.loads(report_path.read_text(encoding='utf-8'))
    display(pd.json_normalize(dataset_report).T.rename(columns={0:'recorded_value'}).head(45))
    sr_csv=ROOT/'results'/'sample_rate_distribution.csv'
    if sr_csv.exists(): display(pd.read_csv(sr_csv))
else:
    print('Full-audit report not present. Enable RUN_FULL_AUDIO_AUDIT for new evidence.')

### Real PyTorch Dataset/DataLoader proof

The three subsets are built using **different predefined folds**, not a random split. For Windows compatibility, begin with `num_workers=0` and small batches. Training can shuffle; validation/test cannot. The code below **decodes examples** but does **not** update model weights or evaluate accuracy.

In [ ]:
if df is not None:
    train_loader,val_loader,test_loader=make_loaders(df,test_fold=TEST_FOLD,
                                   batch_size=2,num_workers=0,seed=67)
    for tag,loader in [('train',train_loader),('validation',val_loader)]:
        x,y=next(iter(loader))
        print(tag, 'waveform:',tuple(x.shape),x.dtype,
              'labels:',y.tolist(),'target type:',y.dtype)
        assert x.shape[1:]==(128000,) and y.dtype==torch.int64
    print('Held-out test DataLoader was constructed but NOT iterated; reserved samples:',len(test_loader.dataset))
    print('This is an input-pipeline check, not an evaluation.')
else:
    print('DataLoader demo skipped: dataset absent.')

## Stage 4 · CNN14 architecture, transfer learning and gradient evidence

The official architecture **retains its AudioSet 527-class layer** (frozen). Our wrapper reads the official **2,048-dimensional embedding** and passes it to an independent randomly initialized `Linear(2048,10)` layer. Logits feed ordinary multi-class **CrossEntropyLoss** (no pre-softmax). Weights are loaded strictly from the tensor-only checkpoint, and the official CNN14 revision is checked.

**Freezing strategies** are controlled ablations under identical data splits and optimization settings:

| Model configuration | Trainable modules | Verified trainable parameters | Role |
|---|---|---:|---|
| `head` | `classifier` | 20,490 | Frozen feature-extraction baseline |
| `fc1` | `backbone.fc1` and classifier | 4,216,842 | Dense fine-tuning baseline |
| `block6` | `backbone.conv_block6`, `backbone.fc1`, classifier | 60,848,138 | Required genuine partial CNN fine-tuning |

All pretrained BatchNorm running means/variances stay frozen even within the trainable later block, an explicit design choice for batch size 1. Only intended trainable parameters enter AdamW. **Actual gradients and parameter updates, not `requires_grad` flags alone, establish fine-tuning.**

In [ ]:
from src.finetune_model import CONFIGS, parameter_summary, load_model, EXPECTED_PANN_COMMIT
from src.precision_policy import choose_precision
print('Freeze configurations:',CONFIGS)
print('Current precision policy:',choose_precision(torch.device('cuda' if torch.cuda.is_available() else 'cpu'),'auto'))
if RUN_UNIT_TESTS:
    run_logged('unit_tests', [sys.executable,'-m','unittest','discover','-s','tests','-v'])
else:
    print('The historical 2026-10-23 Windows run passed 19/19 unit tests. New unit-test run skipped.')

### Strict checkpoint load and model forward (optional)

The historical local verification demonstrated `waveform [1,128000] → official AudioSet scores [1,527], embedding [1,2048] → new raw logits [1,10]` on both CPU and CUDA. Checkpoint loading is memory-intensive and therefore disabled on a default “Run All”. This step executes on **real stored weights** when enabled, not a toy CNN.

In [ ]:
if RUN_CNN14_FORWARD:
    if not SAFE_CKPT.is_file():
        raise FileNotFoundError(f'Missing verified tensor-only weights: {SAFE_CKPT}')
    dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=load_model('block6').to(dev)
    model.eval()
    print('CNN14 + head parameter summary:',parameter_summary(model))
    dummy=torch.zeros((1,128000),dtype=torch.float32,device=dev)
    with torch.inference_mode():
        official=model.backbone(dummy)
        out=model.classifier(official['embedding'])
    print('Official AudioSet scores:',tuple(official['clipwise_output'].shape))
    print('Official embedding:',tuple(official['embedding'].shape))
    print('Adapted raw logits:',tuple(out.shape),'finite:',bool(torch.isfinite(out).all()))
    assert official['clipwise_output'].shape==(1,527)
    assert official['embedding'].shape==(1,2048)
    assert out.shape==(1,10) and torch.isfinite(out).all()
    del model, dummy, official, out
    if dev.type=='cuda': torch.cuda.empty_cache()
else:
    print('CNN14 forward was verified in prior Windows logs; new model load skipped by default.')

### Genuine CUDA backward/update checks — recorded result and reproducibility

**Verified Windows run:** `results/stage4/checks/20261023_071516_424568/summary.json` (19 unit tests passed; all three real-backward configurations passed). The `block6` check observed `conv_block6.conv1.weight` gradient norm **0.224071**, an actual convolution-weight difference of approximately **1.001358e-5** after the optimizer step, and peak allocated CUDA memory **1,257.5 MiB**. Frozen pretrained weights and BatchNorm running statistics did not change. The checked audio was **training-fold only**.

**Important nuance:** a dog-bark clip initially produced `fc1` input RMS **0.0**, so `fc1.weight` and `conv_block6` gradients were zero; an air-conditioner clip yielded nonzero gradients. This observation is open for follow-up during longer training. The script probes up to 12 deterministic training-only clips and does not claim all clips activate the layers.

In [ ]:
last_gradient_run=ROOT/'results'/'stage4'/'checks'/'20261023_071516_424568'/'summary.json'
if last_gradient_run.is_file():
    recorded=json.loads(last_gradient_run.read_text(encoding='utf-8'))
    print('Historical verified Stage 4 run:',last_gradient_run.relative_to(ROOT))
    print('Status:',recorded.get('status'))
    display(pd.DataFrame(recorded.get('steps',[])))
else:
    print('Historical Stage 4 JSON was not copied to this checkout; original observed values are documented above.')
if RUN_REAL_STAGE4_GRADIENT_CHECKS:
    run_logged('stage4_real_backward_checks', [sys.executable,'run_stage4_checks.py'],live=True)

### Diagnostic four-batch pilots (separate from final metrics)

The Stage 4 pilot command runs unit tests and three gradient checks first, then **head** and **block6** one-epoch trials using **four training batches and four validation batches**. A pilot may verify data-flow, checkpoint format, memory and loss finiteness, but **its metrics are NOT valid benchmark performance**. It does not touch the held-out test fold.

In [ ]:
if RUN_DIAGNOSTIC_PILOTS:
    run_logged('stage4_diagnostic_pilots',
               [sys.executable,'run_stage4_checks.py','--pilots'],live=True)
else:
    print('Stage 4 pilots not executed by this notebook run.')

### Full-fold training: validation-selected checkpoints only (intentional opt-in)

The following cell invokes the exact verified modular training implementation from the notebook. It uses **eight official training folds**, **one validation fold**, and **one untouched test fold**. Common settings across the three configurations: seed 67, AdamW, classifier LR `1e-3`, pretrained-layer LR `1e-5`, weight decay `1e-4`, batch size 1, accumulation 4, six *maximum* epochs, early stopping patience 2; BF16 when supported, otherwise FP32. There is **no scheduler** in the present implementation. Training-mode SpecAugment/dropout come from the official model; the dataset module adds no augmentation.

The saving code writes `config.json`, `history.csv`, `learning_curves.png`, validation-selected `best.pt`, and `summary.json` to a unique `results/stage4/` subfolder. Each summary includes `test_evaluated=false`. **Do not use held-out test folds to choose the best configuration.** Execute only after the limited pilot passes and you are satisfied with the learning conditions.

In [ ]:
if RUN_FULL_FOLD_TRAINING:
    assert not RUN_DIAGNOSTIC_PILOTS, 'Do not run pilot and full training in one notebook pass.'
    for config in TRAIN_CONFIGURATIONS:
        run_logged(f'full_fold1_{config}',
            [sys.executable,'-m','src.finetune_train',
             '--config',config,'--test-fold',str(TEST_FOLD),
             '--epochs',str(EPOCHS),'--patience',str(PATIENCE),
             '--batch-size',str(BATCH_SIZE),'--accum-steps',str(ACCUM_STEPS),
             '--amp',AMP_MODE], live=True)
else:
    print('FULL TRAINING DISABLED: no epochs, held-out test scoring or accuracy claims produced.')

## Experiment viewer · Read **actual saved** histories, curves and validation summaries

Unlike temporary terminal scrollback, these records persist across shutdown. A `diagnostic_only` experiment must be excluded from scientific claims. Only a completed full-fold run can support a training-curve discussion; even then it is **validation evidence**, not a held-out test score.

In [ ]:
stage4_root=ROOT/'results'/'stage4'
run_rows=[]
if stage4_root.is_dir():
    for summary_path in sorted(stage4_root.glob('*/summary.json')):
        if summary_path.is_file():
            s=json.loads(summary_path.read_text(encoding='utf-8'))
            if 'config' not in s: continue
            run_rows.append({
                'run_dir':str(summary_path.parent.relative_to(ROOT)),
                'config':s.get('config'),'test_fold_reserved':s.get('test_fold'),
                'diagnostic_only':s.get('diagnostic_only'),
                'epochs_completed':s.get('epochs_completed'),
                'best_val_macro_f1':s.get('best_val_macro_f1'),
                'best_val_accuracy':s.get('best_val_accuracy'),
                'test_evaluated':s.get('test_evaluated',False),
                'peak_trainable_parameters':s.get('parameter_counts',{}).get('trainable')})
runs_df=pd.DataFrame(run_rows)
if runs_df.empty:
    print('No Stage 4 training summary found. The real one-step checks are not training epochs.')
else:
    display(runs_df)
    runs_df.to_csv(EVIDENCE/'training_runs_inventory.csv',index=False)

In [ ]:
# Explore learning curves for a genuinely completed full-fold run, never an old ESC-50 run.
if not runs_df.empty:
    full=runs_df[runs_df['diagnostic_only'].eq(False)].copy()
    if not full.empty:
        selected_dir=ROOT/full.iloc[-1]['run_dir']
        hist_path=selected_dir/'history.csv'
        if hist_path.is_file():
            history=pd.read_csv(hist_path)
            display(history.tail(10))
            fig,axs=plt.subplots(1,2,figsize=(11,4))
            axs[0].plot(history.epoch,history.train_loss,label='Train')
            axs[0].plot(history.epoch,history.val_loss,label='Validation')
            axs[0].set(xlabel='Epoch',ylabel='Cross-entropy',title='Loss')
            axs[1].plot(history.epoch,history.train_accuracy,label='Train (online)')
            axs[1].plot(history.epoch,history.val_accuracy,label='Validation')
            axs[1].set(xlabel='Epoch',ylabel='Accuracy',title='Accuracy',ylim=(0,1))
            for ax in axs:ax.legend();ax.grid(alpha=.2)
            fig.suptitle(f"Observed training: {selected_dir.name}")
            fig.tight_layout();fig.savefig(EVIDENCE/'latest_training_curves.png',dpi=140)
            display(fig);plt.close(fig)
    else:
        print('Only diagnostic pilots found; cannot claim full-fold convergence.')
else:
    print('Learning curves will appear here after a completed full-fold run.')

## CO3 / CO4 / CO5 evidence board (no fabricated numbers)

**CO3 (10 marks):** preprocessing + model integration and gradient update are verified. Training, selected-model evaluation, per-class precision/recall/F1, specificity, multiclass ROC-AUC (from probabilities), and confusion matrix **remain pending**. Future Stage 5 evaluation must load the exact validation-selected checkpoint and evaluate the reserved test fold only after model selection.

**CO4 (5 marks):** published methods must genuinely use **UrbanSound8K**. Include method/year, architecture, external pretraining, fold protocol, reported metrics and original citations. **Do not compare unlike protocols as if directly equivalent.** No claimed SOTA scores are supplied here.

**CO5 (5 marks):** measured *training* comparisons of `head` vs `fc1` vs `block6` are pending; report macro-F1, accuracy, generalization gap, trainable parameters, time, VRAM and fold stability. This is an *experimentally justified layer-freezing study*, not a newly invented algorithm.

In [ ]:
# Only summarize full-fold validation evidence that genuinely exists.
if not runs_df.empty:
    valid=runs_df[(runs_df['diagnostic_only']==False)&
                  (runs_df['test_evaluated']==False)].copy()
    if not valid.empty:
        print('OBSERVED VALIDATION RESULTS ONLY (NOT TEST RESULTS):')
        display(valid[['config','test_fold_reserved','epochs_completed',
                       'best_val_macro_f1','best_val_accuracy','peak_trainable_parameters','run_dir']])
    else:
        print('No completed valid full-fold training runs yet; CO5 numerical comparison pending.')
else:
    print('CO3 model performance, CO4 and CO5 quantitative results: PENDING.')

## Persistent project diary and root README update

This cell creates a dated `PROJECT_NOTEBOOK_PROGRESS.md` record for the current notebook execution and synchronizes a **bounded, clearly marked section** of the root README. Previous README history remains intact. It records which checks were **actually executed in this notebook run** and which artifacts exist; it does not infer model accuracy from the gradient test.

Saved notebook outputs remain in the `.ipynb` after pressing **Ctrl+S**; JSON/CSV/PNG/logs remain under `results/notebook_evidence/`. The latter directory is Git-ignored for size and privacy. Commit the notebook and README to Git to preserve curated progress.

In [ ]:
status_lines=[
    '### Latest notebook execution',
    f'- Notebook session: `{RUN_ID}`',
    f'- Python: `{device_info["python"]}`; PyTorch `{versions.get("torch")}`; CUDA available: `{device_info["cuda_available"]}`',
    f'- Local audio metadata available: `{df is not None}`',
    f'- Tensor-only official CNN14 weights available: `{SAFE_CKPT.is_file()}`',
    f'- Programmatic notebook commands run this session: `{len(notebook_steps)}`',
    '- **Historic verified state:** 8,732 WAV files passed the full decode audit; CNN14 and all three one-step gradient/update modes passed on Windows (23 Oct 2026).',
    '- **Known limitation:** five `fsID` values span official folds; some time intervals overlap.',
    '- **Open work:** multi-batch pilot, full-fold training, held-out metrics, genuine CO4/CO5 comparison, and final packaging (unless subsequent dated evidence establishes them).',
]
if notebook_steps:
    status_lines += ['', '| Notebook step | Exit | Saved log |','|---|---:|---|']
    status_lines += [f'| {s["name"]} | {s["exit_code"]} | `{s["log"]}` |' for s in notebook_steps]
if not runs_df.empty:
    n_diagnostics=int(runs_df['diagnostic_only'].eq(True).sum())
    n_full=int(runs_df['diagnostic_only'].eq(False).sum())
    status_lines.append(f'- Saved Stage 4 training summaries found: `{n_full}` full, `{n_diagnostics}` diagnostic. These are not held-out test results.')
status_text='\n'.join(status_lines)+'\n'
(EVIDENCE/'PROJECT_NOTEBOOK_PROGRESS.md').write_text(status_text,encoding='utf-8')
if UPDATE_README_FROM_NOTEBOOK:
    readme_path=ROOT/'README.md'
    if readme_path.is_file():
        start='<!-- AUTO_NOTEBOOK_STATUS_START -->'
        stop='<!-- AUTO_NOTEBOOK_STATUS_END -->'
        readme_text=readme_path.read_text(encoding='utf-8')
        section=start+'\n'+status_text+stop
        if start in readme_text and stop in readme_text:
            left=readme_text.split(start,1)[0]
            right=readme_text.split(stop,1)[1]
            readme_text=left+section+right
        else:
            readme_text=readme_text+'\n\n'+section+'\n'
        readme_path.write_text(readme_text,encoding='utf-8')
        print('README updated without replacing the existing project history.')
    else:
        print('README not found; wrote session diary only.')
display(Markdown(status_text))

## Optional · Safely clear Python bytecode caches before a clean rerun

**This is NOT required for normal notebook restart.** Python normally refreshes stale bytecode automatically. To remove local Python caches, set `CLEAR_PYTHON_CACHES=True` in the controls and run **only this cell while no training is active**. It deletes `__pycache__`, `.pyc`, and `.pytest_cache` **only** beneath project root `src/`, `tests/` and root itself. It does **not** delete `data/`, `checkpoints/`, `external/`, notebooks, results, figures, Git, or virtual environments. After clearing, select **Kernel → Restart** and rerun the notebook from the top.

`.gitignore` separately prevents these caches from being pushed to GitHub.

In [ ]:
if CLEAR_PYTHON_CACHES:
    removed=[]
    clean_roots=[ROOT,ROOT/'src',ROOT/'tests']
    # Explicit, shallow search within OUR Python code only, never recursive through data/ or external/.
    for parent in clean_roots:
        if not parent.is_dir(): continue
        for d in [parent/'__pycache__',parent/'.pytest_cache']:
            if d.is_dir():
                shutil.rmtree(d)
                removed.append(str(d.relative_to(ROOT)))
        for pyc in parent.glob('*.pyc'):
            pyc.unlink();removed.append(str(pyc.relative_to(ROOT)))
    print('Removed local Python cache items:',removed if removed else 'none')
    print('Restart the Jupyter kernel to rebuild imports cleanly.')
else:
    print('Cache cleanup disabled. To opt in, set CLEAR_PYTHON_CACHES=True.')

### Git cleanliness: verify tracked caches

`.gitignore` blocks **newly added** Python caches, but Git will keep tracking files that were committed before `.gitignore` was added. This notebook checks those paths without deleting any source code or data. The optional `git rm --cached` command untracks only generated cache files from Git's **index** and keeps them on your disk. Never run it blindly against datasets, checkpoints, or model files.

In [ ]:
# Defaults to inspect-only; no Git staging or removal.
UNTRACK_PREVIOUSLY_COMMITTED_CACHES = False
root_git = ROOT/'.git'
if not root_git.exists():
    print('This folder is not itself a Git repository; .gitignore will apply after git init/clone.')
else:
    tracked = subprocess.run(['git','-C',str(ROOT),'ls-files','-z'],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE,check=True).stdout
    tracked_paths = [s.decode('utf-8',errors='replace') for s in tracked.split(b'\x00') if s]
    unwanted = [p for p in tracked_paths if ('__pycache__/' in p.replace('\\','/')
                 or p.endswith(('.pyc','.pyo')) or '.ipynb_checkpoints/' in p)]
    print('Previously tracked generated cache paths:', unwanted[:40])
    if len(unwanted)>40: print('... plus',len(unwanted)-40,'more')
    if UNTRACK_PREVIOUSLY_COMMITTED_CACHES and unwanted:
        run_logged('remove_tracked_cache_files_from_git_index',
                   ['git','-C',str(ROOT),'rm','--cached','--',*unwanted])
        print('Removed from Git index only; local files remain. Review git status before committing.')


## How to continue the project after this notebook

**Next verified-action sequence** in this notebook: enable `RUN_DIAGNOSTIC_PILOTS` (all scripts and weights must be present) and run its cell. If it passes, disable the pilot, enable `RUN_FULL_FOLD_TRAINING` and execute matching `head`, `fc1`, `block6` experiments. The subsequent experiment-viewer cells will show *real saved* CSVs and learning curves after execution. Never treat diagnostic metrics as model accuracy.

**Future additions:** Stage 5 should implement checkpoint-based held-out fold evaluation with accuracy/macro/weighted/per-class F1, precision, recall, one-vs-rest specificity and ROC-AUC from probabilities, plus a confusion matrix; Stage 6 should cite published UrbanSound8K methods; Stage 7 should analyze the controlled ablation, compute costs and, where feasible, multiple folds. These are **pending** and are intentionally not invented in this notebook.

**Before final submission:** confirm the latest teacher instructions, record student name/registration/section in the README, export genuine figures and logs, package the notebook/source/requirements, and name the final ZIP **`RegistrationNumber.zip`** (your actual registration number, not a placeholder). Exclude huge local audio and weights unless instructions explicitly require them; document their acquisition and checksum. Do not push private filesystem paths or user data publicly without reviewing saved cell outputs.

### Sources and provenance
- [Official PANNs implementation](https://github.com/qiuqiangkong/audioset_tagging_cnn)
- [Published PANNs CNN14 AudioSet checkpoint](https://zenodo.org/records/3576403)
- [UrbanSound8K dataset](https://urbansounddataset.weebly.com/urbansound8k.html)
- Original teacher assignment: `project_submission_instructions` (project source, verify against any updated assignment).